# 06 · E4: control runs for the Baseline → Slakh-FT comparison

**Question.** Slakh-FT improved over the Baseline, but the fine-tune changed three things at once (notebook 00): more training (25,625 steps with a new learning-rate schedule), more data (1229 instead of 704 Slakh tracks) and different codebook loss weights. The review asked to separate the effect of the data from that of the loss weights.

**Method.** Two control runs start from the Baseline weights with exactly the Slakh-FT procedure (25,625 steps, LR 5e-5 → 1e-5 with 500 warmup steps, batch 24, 20 crops per track, AdamW, fp32), changing one factor at a time:

| Run | Training tracks | Loss weights | Compared with | Isolates |
|---|---|---|---|---|
| R1 | 704 (the Baseline's) | Baseline [4, 3.2, 2.7, 2.2, 2, 2, 2, 2] | Baseline | extra training only |
| R2 | 1229 (all) | Baseline | R1 | the new data |
| Slakh-FT (existing) | 1229 | [1, 1, 2, 2, 2.5, 2.5, 2.5, 2.5] | R2 | the new loss weights |

R2 is the more important run (Slakh-FT − R2 is the effect of the weights); R1 completes the decomposition. Both are evaluated on the Slakh test split exactly like Table 1 (notebook 01) and compared with the unweighted per-codebook cross-entropy, which does not depend on the loss weights of a run.

**Sessions.** One run takes several hours (the log prints the measured seconds per step and the remaining time). The loop stops by itself after `TIME_BUDGET_HOURS`, saving `last.pt`; to continue, add this notebook's previous output as an input and run it again: the run resumes exactly where it stopped (`src/training_loop.py`, `tests/test_training_loop.py`).

**Kaggle settings:** GPU (T4 or P100), Internet on, **Save & Run All**. **Inputs:** `slakh-tensors-final`, `secunda-slakh-60-epochs` (Baseline), and — only when resuming — the output of the previous version of this notebook.

In [1]:
# ── Repository bootstrap: make src/ importable on Kaggle and locally ──
# On Kaggle the repository is cloned into /kaggle/working (Internet must be
# enabled in the notebook settings). Locally, set SECUNDA_REPO_DIR to the
# repository root and nothing is cloned.
import os
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/DanBracs7/Adaptive-Secunda-Synth.git"
REPO_BRANCH = "revision"  # switch to "main" after merging the revision
REPO_DIR = Path(os.environ.get("SECUNDA_REPO_DIR", "/kaggle/working/Adaptive-Secunda-Synth"))

if not (REPO_DIR / "src").exists():
    subprocess.run(
        ["git", "clone", "--depth", "1", "--branch", REPO_BRANCH, REPO_URL, str(REPO_DIR)],
        check=True,
    )

sys.path.insert(0, str(REPO_DIR / "src"))

commit = subprocess.run(
    ["git", "-C", str(REPO_DIR), "rev-parse", "--short", "HEAD"],
    capture_output=True,
    text=True,
).stdout.strip()

print("Repository:", REPO_DIR)
print("Commit:", commit or "unknown (not a git checkout)")

Cloning into '/kaggle/working/Adaptive-Secunda-Synth'...


Repository: /kaggle/working/Adaptive-Secunda-Synth
Commit: 12f5cae


In [2]:
# ── Choose the run ──
RUN_NAME = "R1"            # "R2" first, then "R1"
TIME_BUDGET_HOURS = 11.0   # Kaggle stops a session after 12 hours

# Overrides used only to test the notebook quickly on small data.
TIME_BUDGET_HOURS = float(os.environ.get("SECUNDA_E4_TIME_BUDGET_HOURS", TIME_BUDGET_HOURS))
MAX_STEPS = int(os.environ.get("SECUNDA_E4_MAX_STEPS", 25_625))

In [3]:
import json

import numpy as np
import torch
from torch.utils.data import DataLoader

from audio_dataset import SecundaAudioDataset
from frame_codec import NUM_CODEBOOKS, CodebookSerializer
from paths import get_path, source_roots_for_split
from training_loop import build_optimizer, filter_rows, train
from training_utils import evaluate_on_loader, load_checkpoint_model

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu")

BASELINE_LOSS_WEIGHTS = [4.0, 3.2, 2.7, 2.2, 2.0, 2.0, 2.0, 2.0]


def baseline_tracks_only(row):
    # The Baseline was trained on the train split of slakh/train_000-799 (704 tracks).
    return int(Path(row["path"]).stem.split("_")[-1]) <= 799


RUNS = {
    "R1": {"keep": baseline_tracks_only, "expected_tracks": 704},
    "R2": {"keep": lambda row: True, "expected_tracks": 1229},
}
assert RUN_NAME in RUNS, RUN_NAME

# Same procedure as the Slakh-FT run (its stored config, notebook 00),
# except the loss weights, which stay those of the Baseline.
TRAINING_CONFIG = {
    "batch_size": 24,
    "num_workers": 2,
    "samples_per_track": 20,
    "segment_seconds": 10.0,
    "frame_rate": 75,
    "peak_lr": 5e-5,
    "min_lr": 1e-5,
    "warmup_steps": 500,
    "max_steps": MAX_STEPS,          # 25,625 = the Slakh-FT run
    "weight_decay": 0.01,
    "grad_clip_norm": 1.0,
    "codebook_loss_weights": BASELINE_LOSS_WEIGHTS,
    "seed": 42,
    "save_every_steps": 500,
    "log_every_steps": 100,
    "training_stage": f"e4_{RUN_NAME}_from_baseline",
}

OUTPUT_DIR = get_path("output_dir") / f"e4_{RUN_NAME}"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

Device: Tesla T4


In [4]:
train_dataset = SecundaAudioDataset(
    manifest_path=get_path("manifest"),
    source_roots=source_roots_for_split("train"),
    split="train",
    stage="pretrain",
    segment_seconds=TRAINING_CONFIG["segment_seconds"],
    frame_rate=TRAINING_CONFIG["frame_rate"],
    fixed_start=False,
    seed=TRAINING_CONFIG["seed"],
    samples_per_track=TRAINING_CONFIG["samples_per_track"],
)
filter_rows(train_dataset, RUNS[RUN_NAME]["keep"])

val_dataset = SecundaAudioDataset(
    manifest_path=get_path("manifest"),
    source_roots=source_roots_for_split("val"),
    split="val",
    stage="pretrain",
    segment_seconds=TRAINING_CONFIG["segment_seconds"],
    frame_rate=TRAINING_CONFIG["frame_rate"],
    fixed_start=True,
    seed=TRAINING_CONFIG["seed"],
    samples_per_track=1,
)
val_loader = DataLoader(val_dataset, batch_size=TRAINING_CONFIG["batch_size"], shuffle=False,
                        num_workers=TRAINING_CONFIG["num_workers"], pin_memory=True)

print(f"{RUN_NAME}: {len(train_dataset.rows)} training tracks, {len(train_dataset)} crops per epoch, "
      f"{-(-len(train_dataset) // TRAINING_CONFIG['batch_size'])} steps per epoch")
print("Validation tracks (all Slakh val, same for both runs):", len(val_dataset.rows))
if len(train_dataset.rows) != RUNS[RUN_NAME]["expected_tracks"]:
    print(f"WARNING: expected {RUNS[RUN_NAME]['expected_tracks']} tracks (incomplete dataset?)")

R1: 704 training tracks, 14080 crops per epoch, 587 steps per epoch
Validation tracks (all Slakh val, same for both runs): 166


In [5]:
# Start from the Baseline weights with a fresh optimizer, as the Slakh-FT run did,
# or resume from the last.pt of a previous session of this notebook.
resume_candidates = sorted(Path("/kaggle/input").rglob(f"e4_{RUN_NAME}/last.pt")) if Path("/kaggle/input").exists() else []
resume_path = Path(os.environ["SECUNDA_E4_RESUME"]) if "SECUNDA_E4_RESUME" in os.environ else (
    resume_candidates[-1] if resume_candidates else None)

model, baseline_checkpoint = load_checkpoint_model(get_path("ckpt_baseline"), device)
model_config = {key: baseline_checkpoint["config"][key] for key in
                ("embedding_dim", "num_layers", "num_heads", "feedforward_dim", "dropout")}
model_config["max_sequence_length"] = model.max_sequence_length
optimizer = build_optimizer(model, TRAINING_CONFIG["peak_lr"], TRAINING_CONFIG["weight_decay"])

print("Start:", f"resume from {resume_path}" if resume_path else "Baseline weights, fresh optimizer")

/kaggle/working/Adaptive-Secunda-Synth/src/model.py:107: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.transformer = nn.TransformerEncoder(


Start: resume from /kaggle/input/notebooks/danielebracoloni/notebooks06-e4-control-runs/e4_R1/last.pt


In [6]:
state = train(
    model=model,
    optimizer=optimizer,
    train_dataset=train_dataset,
    val_loader=val_loader,
    serializer=CodebookSerializer(),
    device=device,
    config={**model_config, **TRAINING_CONFIG},
    output_dir=OUTPUT_DIR,
    resume_path=resume_path,
    time_budget_seconds=TIME_BUDGET_HOURS * 3600,
)
print("Finished:", state["finished"], "| step", state["global_step"], "of", TRAINING_CONFIG["max_steps"])

Resumed from /kaggle/input/notebooks/danielebracoloni/notebooks06-e4-control-runs/e4_R1/last.pt: epoch 20, batch 403, step 11556
step 11600/25625 | epoch 20 batch 447/587 | loss 4.7393 | lr 3.36e-05 | 2.68 s/step | remaining 10.4 h
step 11700/25625 | epoch 20 batch 547/587 | loss 4.7099 | lr 3.34e-05 | 2.54 s/step | remaining 9.8 h
epoch 20 done at step 11740 | train 4.6992 | val 4.8257
step 11800/25625 | epoch 21 batch 60/587 | loss 4.8238 | lr 3.31e-05 | 2.63 s/step | remaining 10.1 h
step 11900/25625 | epoch 21 batch 160/587 | loss 4.4951 | lr 3.29e-05 | 2.59 s/step | remaining 9.9 h
step 12000/25625 | epoch 21 batch 260/587 | loss 4.7084 | lr 3.26e-05 | 2.57 s/step | remaining 9.7 h
step 12100/25625 | epoch 21 batch 360/587 | loss 4.8675 | lr 3.24e-05 | 2.55 s/step | remaining 9.6 h
step 12200/25625 | epoch 21 batch 460/587 | loss 4.7554 | lr 3.22e-05 | 2.54 s/step | remaining 9.5 h
step 12300/25625 | epoch 21 batch 560/587 | loss 4.7004 | lr 3.19e-05 | 2.54 s/step | remaining 9.4 

## Evaluation on the Slakh test split (only when the run is finished)

In [7]:
# Table 1 protocol (notebook 01): 151 test tracks, one fixed 10-s crop each.
# The Baseline and Slakh-FT values are those printed by notebook 01.
REPORT_LOSS_WEIGHTS = [1.0, 1.0, 2.0, 2.0, 2.5, 2.5, 2.5, 2.5]
TABLE1_PER_CODEBOOK = {
    "Baseline": [3.286917209625244, 4.3396148681640625, 4.921988487243652, 5.38573694229126,
                 5.76570987701416, 6.011356830596924, 6.148679733276367, 6.23488187789917],
    "Slakh-FT": [3.0142502784729004, 4.02440071105957, 4.526853084564209, 4.990567207336426,
                 5.38828706741333, 5.711738109588623, 5.966355323791504, 6.128832817077637],
}

if state["finished"]:
    test_dataset = SecundaAudioDataset(
        manifest_path=get_path("manifest"), source_roots=source_roots_for_split("test"),
        split="test", stage="pretrain", segment_seconds=10.0, frame_rate=75,
        fixed_start=True, seed=42, samples_per_track=1,
    )
    test_loader = DataLoader(test_dataset, batch_size=24, shuffle=False, num_workers=2, pin_memory=True)
    final_path = OUTPUT_DIR / "final.pt"
    if not final_path.exists() and resume_path is not None:
        final_path = resume_path.parent / "final.pt"   # finished in a previous session
    final_model, _ = load_checkpoint_model(final_path, device)
    metrics = evaluate_on_loader(final_model, test_loader, CodebookSerializer(), device, REPORT_LOSS_WEIGHTS)
    metrics["unweighted_loss"] = float(np.mean(metrics["per_codebook_loss"]))
    with open(OUTPUT_DIR / "test_metrics.json", "w", encoding="utf-8") as file:
        json.dump(metrics, file, indent=2)

    # Model-only copy (no optimizer state, ~195 MB) for publishing.
    torch.save({"model_state_dict": final_model.state_dict(),
                "config": {**model_config, **TRAINING_CONFIG}},
               OUTPUT_DIR / f"secunda_e4_{RUN_NAME}_final_model.pt")

    print(f"{'':10s}" + "".join(f"{f'Q{q}':>8s}" for q in range(NUM_CODEBOOKS)) + f"{'unweighted':>12s}")
    for name, values in [*TABLE1_PER_CODEBOOK.items(), (RUN_NAME, metrics["per_codebook_loss"])]:
        print(f"{name:10s}" + "".join(f"{value:8.3f}" for value in values) + f"{np.mean(values):12.3f}")
else:
    print("Run not finished: add this version's output as an input and run the notebook again.")

/kaggle/working/Adaptive-Secunda-Synth/src/model.py:107: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.transformer = nn.TransformerEncoder(


                Q0      Q1      Q2      Q3      Q4      Q5      Q6      Q7  unweighted
Baseline     3.287   4.340   4.922   5.386   5.766   6.011   6.149   6.235       5.262
Slakh-FT     3.014   4.024   4.527   4.991   5.388   5.712   5.966   6.129       4.969
R1           2.923   3.940   4.518   5.037   5.492   5.826   6.054   6.168       4.995


### How to read the result

With R1 and R2 finished, the change from Baseline to Slakh-FT splits into three parts (unweighted test cross-entropy):
- **extra training** = R1 − Baseline;
- **new data** = R2 − R1;
- **new loss weights** = Slakh-FT − R2 (per codebook: the new weights lower the emphasis on Q0–Q1 and raise it on Q4–Q7, so they may trade coarse for fine codebooks).

One run per configuration: differences of a few hundredths are within run-to-run noise and should not be over-interpreted.

### Results of this run (R1)

**Training.** 25,625 steps on a T4 in two sessions. The first session (9 h) stopped at step 11,556 (epoch 20, batch 403); the second resumed exactly there (the learning rate at step 11,600, 3.36e-5, is the value of the schedule) and finished at 2.50 s/step. This notebook contains the log of the second session only; the full history of both sessions is in `results/e4/R1/history.json`. The validation loss (Baseline loss weights, 166 Slakh validation tracks) decreased at every one of the 44 epochs, from 4.990 to 4.767, with no sign of overfitting even though R1 sees the same 704 tracks again and again. (R2, same weights and validation set, ended at 4.704.)

**Test (Table 1 protocol).** Unweighted cross-entropy: Baseline 5.262 → **R1 4.995** → R2 4.934 → Slakh-FT 4.969.

**Decomposition of the Baseline → Slakh-FT improvement (−0.293):**

| Step | Δ unweighted | Share | Q0 | Q1 | Q2 | Q3 | Q4 | Q5 | Q6 | Q7 |
|---|---|---|---|---|---|---|---|---|---|---|
| extra training (R1 − Baseline) | −0.267 | 91% | −0.364 | −0.400 | −0.404 | −0.349 | −0.274 | −0.186 | −0.094 | −0.067 |
| new data (R2 − R1) | −0.061 | 21% | −0.075 | −0.085 | −0.088 | −0.078 | −0.060 | −0.049 | −0.029 | −0.020 |
| new loss weights (Slakh-FT − R2) | +0.035 | −12% | +0.167 | +0.169 | +0.097 | +0.031 | −0.043 | −0.065 | −0.059 | −0.019 |

**Interpretation.**
- Most of the improvement comes from training longer on the same data: the Baseline had not converged, and R1 alone improves almost as much as Slakh-FT. "Extra training" includes both the additional steps and the new learning-rate schedule, which the fine-tune changed together.
- The new data add a smaller improvement, of the same sign on every codebook.
- The new loss weights move accuracy from Q0–Q3 to Q4–Q7; on the unweighted average the trade is slightly negative.
- One run per configuration: the data and weight effects (a few hundredths) could change with another training seed. Notebook 07 adds per-segment bootstrap confidence intervals over the test segments.